# EJERCICIOS

In [1]:
import warnings
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier,StackingClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

In [2]:
warnings.filterwarnings("ignore", category=FutureWarning)

## EJERCICIO 1

Si has entrenado cinco modelos diferentes en el mismo conjunto de entrenamiento exacto y todos consiguen una precisión del 95%, ¿hay alguna posibilidad de que puedas combinar estos modelos para obtener mejores resultados? 

Si la respuesta es sí, ¿cómo? Si la respuesta es no, ¿por qué?

In [ ]:
# Sí, el 5% de error restante puede ser diferente en cada modelo, es decir, uno de ellos puede fallar con el registro 1, mientras que los demás lo hagan bien, otro con el 2 y así sucesivamente

## EJERCICIO 2

Carga el conjunto de datos MNIST y divídelo en un conjunto de entrenamiento, un conjunto de validación y un conjunto de prueba (por ejemplo, utiliza 50.000 instancias para entrenamiento, 10.000 para validación y 10.000 para pruebas). 

Después, entrena varios clasificadores diferentes (uno de ellos que sea un árbol de decisión). 

A continuación, intenta combinarlos en un ensamble que supere en rendimiento a cada clasificador individual del conjunto de validación, utilizando hard voting. 

Una vez que hayas encontrado uno, pruébalo en el conjunto de pruebas.

In [6]:
mnist = fetch_openml('mnist_784', as_frame=False)

X_train, y_train = mnist.data[:50_000], mnist.target[:50_000]
X_val,   y_val   = mnist.data[50_000:60_000], mnist.target[50_000:60_000]
X_test,  y_test  = mnist.data[60_000:], mnist.target[60_000:]

print(X_train.shape, X_val.shape, X_test.shape)

(50000, 784) (10000, 784) (10000, 784)


In [7]:
decision_tree = DecisionTreeClassifier(random_state=42)
random_forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
kneighbors = KNeighborsClassifier(n_jobs=-1)
logistic_regression = LogisticRegression(max_iter=200)
svc = SVC(random_state=42)

modelos = [("decision_tree", decision_tree), ("random_forest", random_forest), ("kneighbors", kneighbors), ("logistic_regression", logistic_regression), ("svc", svc)]

for nombre, modelo in modelos:
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_val)
    print(nombre, accuracy_score(y_val, pred))

decision_tree 0.8822
random_forest 0.9736
kneighbors 0.9718


c:\Users\Raul\Documents\Ra\Clase - IABD\SAPA\IABD-SAPA-Aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logistic_regression 0.9258
svc 0.9802


In [8]:
modelos_ensamble = [("random_forest", random_forest), ("kneighbors", kneighbors), ("svc", svc)]

voting = VotingClassifier(modelos, voting="hard")
voting.fit(X_train, y_train)
print("ensamble", accuracy_score(y_val, voting.predict(X_val)))

c:\Users\Raul\Documents\Ra\Clase - IABD\SAPA\IABD-SAPA-Aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


ensamble 0.9778


In [10]:
print("ensamble sobre test", accuracy_score(y_test, voting.predict(X_test)))

ensamble sobre test 0.9722


## EJERCICIO 3

Ejecuta los clasificadores individuales del ejercicio anterior para hacer predicciones en el conjunto de entrenamiento y crea un nuevo conjunto de entrenamiento con las predicciones resultantes: cada instancia de entrenamiento es un vector que contiene el conjunto de predicciones de todos tus clasificadores para una imagen y el objetivo es la clase de la imagen. Entrena un clasificador (RandomForestClassifier) en este nuevo conjunto de entrenamiento. 

Acabas de entrenar un blender y, junto a los clasificadores, forma un ensamble de stacking.

Ahora, evalúa el ensamble en el conjunto de prueba. 

¿Cómo es en comparación con el clasificador de votación que has entrenado antes?

Haz lo mismo usando StackingClassifier

In [9]:
# Predicciones de los modelos ya entrenados sobre validación
X_blend_train = np.column_stack([m.predict(X_val) for _, m in modelos])

blender = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
blender.fit(X_blend_train, y_val)

# Evaluación en test: primero predicen los modelos base, luego el blender
X_blend_test = np.column_stack([m.predict(X_test) for _, m in modelos])
print("stacking manual", accuracy_score(y_test, blender.predict(X_blend_test)))

# Comparación con el voting
print("voting", accuracy_score(y_test, voting.predict(X_test)))


stacking = StackingClassifier(
    estimators=modelos,
    final_estimator=RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    cv=3,
    stack_method="predict",
)
stacking.fit(X_train, y_train)
print("StackingClassifier", accuracy_score(y_test, stacking.predict(X_test)))

stacking manual 0.9745
voting 0.9722


c:\Users\Raul\Documents\Ra\Clase - IABD\SAPA\IABD-SAPA-Aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Raul\Documents\Ra\Clase - IABD\SAPA\IABD-SAPA-Aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You migh

StackingClassifier 0.9761
